# Phase 2 — **Scheduled QAT** (Main Contribution) — Google Colab

**Hypothesis.** Standard QAT applies full quantization noise from step 1, forcing the model to handle the largest precision shock immediately. Gradually reducing the simulated bit-width over training — first FP32 → 16 → 8 → 4 — gives the weights time to adapt at each level before the next shock. We expect **lower KL divergence and lower perplexity at the same target bit-width** versus standard QAT.

We compare three schedules: linear (constant slope), cosine (slow→fast→slow), step (hard drops with plateaus).

**Inputs:** `/content/drive/MyDrive/sqat-baseline/results/baseline/fp32_logits.pt`

**Outputs (saved at the end to `/content/drive/MyDrive/sqat-scheduled-qat/`):**
- `models/checkpoints/scheduled_qat_{linear,cosine,step}_int4.pt` — renamed
- `results/scheduled_qat_*_int4/training_results.json`
- `results/scheduled_qat_inference.json`
- `results/logs/scheduled_*_int4/per_step_loss.jsonl` — micro view
- `results/logs/scheduled_*_int4/training_steps.jsonl` — macro view
- `data_samples/{train,val,test}_sample.pt`
- `results/metric_summary.csv`

**Estimated time on Colab T4:** 3 schedules × ~1.7 h ≈ **5 h** for INT4."

## 1. Setup

In [ ]:
import os, sys, subprocess
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

WORKING_DIR  = "/content"
REPO_DIR     = f"{WORKING_DIR}/scheduled-qat-for-slm"
GITHUB_URL   = "https://github.com/JpCurada/scheduled-qat-for-slm.git"
BASELINE_DIR = "/content/drive/MyDrive/sqat-baseline/results/baseline"
DRIVE_ROOT   = "/content/drive/MyDrive/sqat-scheduled-qat"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", GITHUB_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

FP32_LOGITS = Path(BASELINE_DIR) / "fp32_logits.pt"
assert FP32_LOGITS.exists(), (
    f"FP32 logits not found at {FP32_LOGITS}. "
    "Run notebook 01 first."
)
print(f"FP32 logits: {FP32_LOGITS}")

In [ ]:
!pip install -q -e ".[viz]"
import torch
print(f"GPU: {torch.cuda.get_device_name() if torch.cuda.is_available() else 'CPU'}")

## 2. Schedule visualization

Before training, plot what each schedule actually does to the bit-width over time. The transitions are read from the YAML configs and converted to discrete fake-quant levels via `snap_bits()` — that mapping is what the model actually experiences.

Reading this plot:
- The **continuous** lines show the schedule's float bit-width.
- The **stepped** lines show the actual fake-quant level applied (32→None, 16→None, 8→INT8, 4→INT4).

In [ ]:
import yaml
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.quantization.scheduler import build_scheduler, snap_bits
from src.utils.config_loader import load_config

def plot_schedules(name_to_cfg: dict, suptitle: str):
    fig = make_subplots(rows=1, cols=len(name_to_cfg),
                        subplot_titles=list(name_to_cfg.keys()),
                        shared_yaxes=True)
    for col, (name, cfg_path) in enumerate(name_to_cfg.items(), 1):
        cfg = load_config(cfg_path)
        sched = build_scheduler(cfg.schedule, total_epochs=float(cfg.training.epochs))
        epochs = np.linspace(0, cfg.training.epochs, 400)
        cont = [sched.get_state(e).continuous_bits for e in epochs]
        snap = [snap_bits(c) or 32 for c in cont]
        fig.add_trace(go.Scatter(x=epochs, y=cont, name="continuous",
                                 line=dict(color="#4C72B0"), legendgroup=str(col),
                                 showlegend=(col == 1)),
                      row=1, col=col)
        fig.add_trace(go.Scatter(x=epochs, y=snap, name="fake-quant level",
                                 line=dict(color="#C44E52", shape="hv"),
                                 legendgroup=str(col), showlegend=(col == 1)),
                      row=1, col=col)
        fig.update_xaxes(title_text="epoch", row=1, col=col)
    fig.update_yaxes(title_text="bit-width", tickvals=[4, 8, 16, 32], row=1, col=1)
    fig.update_layout(title=suptitle, height=420, hovermode="x unified",
                      margin=dict(t=80, b=40, l=60, r=20))
    fig.show()

ORIGINAL_PATHS = {
    "linear": "configs/scheduled_qat/scheduled_linear_int4.yaml",
    "cosine": "configs/scheduled_qat/scheduled_cosine_int4.yaml",
    "step":   "configs/scheduled_qat/scheduled_step_int4.yaml",
}
plot_schedules(ORIGINAL_PATHS, "Original 3-epoch schedules (start_bits=32 → target_bits=4)")

## 3. Kaggle config overrides

Same overrides as notebook 03 (Standard QAT) so the comparison is apples-to-apples. The `schedule` block is preserved verbatim from the original config — it's the only thing that differs across the three runs.

In [ ]:
import yaml

DRIVE_MODEL_CACHE = "/content/drive/MyDrive/sqat-baseline/models/base"
LOCAL_MODEL_CACHE = f"{REPO_DIR}/models/base"
MODEL_CACHE       = DRIVE_MODEL_CACHE if Path(DRIVE_MODEL_CACHE).exists() else LOCAL_MODEL_CACHE

COLAB_CFG = Path(REPO_DIR) / "configs_colab" / "scheduled_qat"
COLAB_CFG.mkdir(parents=True, exist_ok=True)

RUN_BEST_INT8 = False

KAGGLE_EPOCHS    = 1
ORIGINAL_EPOCHS  = 3
SCHEDULE_SCALE   = KAGGLE_EPOCHS / ORIGINAL_EPOCHS

def patch_sqat_config(src: str, dst: Path) -> Path:
    with open(src) as f:
        cfg = yaml.safe_load(f)
    cfg["model"]["cache_dir"] = MODEL_CACHE
    cfg["data"]["seq_length"] = 512
    cfg["training"]["epochs"] = KAGGLE_EPOCHS
    cfg["training"]["batch_size"] = 4
    cfg["training"]["gradient_accumulation_steps"] = 8
    cfg["training"]["warmup_steps"] = 100
    cfg["logging"]["save_every_steps"] = 999999
    cfg["logging"]["eval_every_steps"] = 500
    cfg["logging"]["log_dir"] = f"{REPO_DIR}/results/logs/{dst.stem}/"
    cfg["export"]["output_dir"] = f"{REPO_DIR}/models/gguf/"

    sch = cfg["schedule"]
    sch["warmup_epochs"]        = round(sch["warmup_epochs"]        * SCHEDULE_SCALE, 4)
    sch["stabilization_epochs"] = round(sch["stabilization_epochs"] * SCHEDULE_SCALE, 4)
    for t in sch.get("transitions") or []:
        t["epoch"] = round(t["epoch"] * SCHEDULE_SCALE, 4)

    with dst.open("w") as f:
        yaml.safe_dump(cfg, f, sort_keys=False)
    return dst

kaggle_cfgs = {}
for name in ("linear", "cosine", "step"):
    src = f"configs/scheduled_qat/scheduled_{name}_int4.yaml"
    dst = COLAB_CFG / f"scheduled_{name}_int4.yaml"
    kaggle_cfgs[name] = patch_sqat_config(src, dst)
    print(f"  {name:7s} -> {dst}")

print(f"\nModel cache: {MODEL_CACHE}")

### Patched schedules — what the Kaggle runs actually use

The plot below shows the schedules after compression to one epoch. Compare to the 3-epoch originals above — same shape, smaller time axis. The model spends proportionally less time at each precision level than in the published configs.

In [ ]:
plot_schedules({k: str(v) for k, v in kaggle_cfgs.items()},
               f"Patched {KAGGLE_EPOCHS}-epoch schedules (Kaggle runs)")

## 4. Linear schedule — INT4

Constant rate of precision reduction. Simplest baseline schedule.

In [ ]:
import gc
from src.training.trainer import run_experiment

results_linear = run_experiment(
    config_path=str(kaggle_cfgs["linear"]),
    device_str="cuda",
    baseline_logits=str(FP32_LOGITS),
    run_benchmarks=False,
)

print("\nLinear schedule INT4:")
for k, v in results_linear.items():
    print(f"  {k:25s}  {v}")

gc.collect(); torch.cuda.empty_cache()

## 5. Cosine schedule — INT4

Slow start, fast middle, slow end. Should beat linear if the cosine LR analogy carries over to precision.

In [ ]:
results_cosine = run_experiment(
    config_path=str(kaggle_cfgs["cosine"]),
    device_str="cuda",
    baseline_logits=str(FP32_LOGITS),
    run_benchmarks=False,
)

print("\nCosine schedule INT4:")
for k, v in results_cosine.items():
    print(f"  {k:25s}  {v}")

gc.collect(); torch.cuda.empty_cache()

## 6. Step schedule — INT4

Hard drops at the configured transition epochs, with flat plateaus between. Each precision level gets a stable training period before the next drop.

In [ ]:
results_step = run_experiment(
    config_path=str(kaggle_cfgs["step"]),
    device_str="cuda",
    baseline_logits=str(FP32_LOGITS),
    run_benchmarks=False,
)

print("\nStep schedule INT4:")
for k, v in results_step.items():
    print(f"  {k:25s}  {v}")

gc.collect(); torch.cuda.empty_cache()

## 7. Schedule comparison (INT4)

The headline table for the thesis. Lower PPL and lower KLD = better. The winning schedule is fed into the optional INT8 cell below.

In [ ]:
import json
import pandas as pd

with open(Path(BASELINE_DIR) / "baseline_results.json") as f:
    fp32 = json.load(f)

rows = [
    {"schedule": "linear", **{k: results_linear.get(k) for k in ("perplexity", "kl_divergence", "final_loss", "training_time_seconds")}},
    {"schedule": "cosine", **{k: results_cosine.get(k) for k in ("perplexity", "kl_divergence", "final_loss", "training_time_seconds")}},
    {"schedule": "step",   **{k: results_step.get(k)   for k in ("perplexity", "kl_divergence", "final_loss", "training_time_seconds")}},
]
df = pd.DataFrame(rows)
df["ppl_delta_pct"] = ((df["perplexity"] / fp32["perplexity"]) - 1.0) * 100
df = df.round(4)
df

In [ ]:
def plot_method_comparison(rows, title, fp32_ppl=None):
    labels = [r["label"] for r in rows]
    fig = make_subplots(rows=1, cols=3,
                        subplot_titles=("Perplexity (lower=better)",
                                        "KL Divergence vs FP32 (lower=better)",
                                        "Model size (GB)"))
    fig.add_trace(go.Bar(x=labels, y=[r["perplexity"] for r in rows],
                         marker_color="#4C72B0",
                         text=[f"{r['perplexity']:.3f}" if r["perplexity"] is not None else "—" for r in rows],
                         textposition="outside"), 1, 1)
    if fp32_ppl is not None:
        fig.add_hline(y=fp32_ppl, line_dash="dash", line_color="black",
                      annotation_text=f"FP32 = {fp32_ppl:.3f}", row=1, col=1)
    fig.add_trace(go.Bar(x=labels, y=[r["kl_divergence"] for r in rows],
                         marker_color="#DD8452",
                         text=[f"{r['kl_divergence']:.4f}" if r["kl_divergence"] is not None else "—" for r in rows],
                         textposition="outside"), 1, 2)
    fig.add_trace(go.Bar(x=labels, y=[r["size_gb"] for r in rows],
                         marker_color="#55A868",
                         text=[f"{r['size_gb']:.2f}" for r in rows],
                         textposition="outside"), 1, 3)
    fig.update_layout(title=title, showlegend=False, height=420,
                      margin=dict(t=80, b=40, l=40, r=20))
    fig.show()

# Three schedules + the FP32 baseline. INT4 only (this notebook's focus).
rows = [
    {"label": "FP32 baseline", "perplexity": fp32.get("perplexity"),               "kl_divergence": 0.0, "size_gb": 6.5},
    {"label": "linear INT4",   "perplexity": results_linear.get("perplexity"),     "kl_divergence": results_linear.get("kl_divergence"), "size_gb": 0.85},
    {"label": "cosine INT4",   "perplexity": results_cosine.get("perplexity"),     "kl_divergence": results_cosine.get("kl_divergence"), "size_gb": 0.85},
    {"label": "step INT4",     "perplexity": results_step.get("perplexity"),       "kl_divergence": results_step.get("kl_divergence"),   "size_gb": 0.85},
]
plot_method_comparison(rows, "Scheduled QAT INT4 — quality by schedule",
                       fp32_ppl=fp32.get("perplexity"))

best = sched_df.iloc[0]["schedule"]
print(f"\nBest schedule (lowest KLD): {best}")

### Training loss curves — three schedules overlaid

If the schedule helps, the curves should split: the schedule that lands at INT4 most gracefully should have the lowest train loss at the end. Step-schedule transitions appear as visible kinks where precision drops; linear/cosine should be smoother.

In [ ]:
def read_jsonl(path):
    if not Path(path).exists():
        return []
    with open(path) as f:
        return [json.loads(l) for l in f if l.strip()]

# Macro view (val PPL at eval intervals)
eval_logs = {
    "linear": read_jsonl(f"{REPO_DIR}/results/logs/scheduled_linear_int4/training_steps.jsonl"),
    "cosine": read_jsonl(f"{REPO_DIR}/results/logs/scheduled_cosine_int4/training_steps.jsonl"),
    "step":   read_jsonl(f"{REPO_DIR}/results/logs/scheduled_step_int4/training_steps.jsonl"),
}
# Micro view (loss at every step)
step_logs = {
    "linear": read_jsonl(f"{REPO_DIR}/results/logs/scheduled_linear_int4/per_step_loss.jsonl"),
    "cosine": read_jsonl(f"{REPO_DIR}/results/logs/scheduled_cosine_int4/per_step_loss.jsonl"),
    "step":   read_jsonl(f"{REPO_DIR}/results/logs/scheduled_step_int4/per_step_loss.jsonl"),
}

fig = make_subplots(specs=[[{"secondary_y": True}]])
colors = {"linear": "#4C72B0", "cosine": "#DD8452", "step": "#55A868"}
for name in ("linear", "cosine", "step"):
    if step_logs[name]:
        fig.add_trace(go.Scatter(
            x=[e["step"] for e in step_logs[name]],
            y=[e["loss"] for e in step_logs[name]],
            name=f"{name} loss (per step)",
            line=dict(color=colors[name], width=1), opacity=0.5,
        ), secondary_y=False)
    if eval_logs[name]:
        fig.add_trace(go.Scatter(
            x=[e["step"]    for e in eval_logs[name]],
            y=[e.get("val_ppl") for e in eval_logs[name]],
            name=f"{name} val ppl",
            line=dict(color=colors[name], dash="dash", width=2),
            mode="lines+markers",
        ), secondary_y=True)

fig.update_xaxes(title_text="step")
fig.update_yaxes(title_text="train loss (per-step)", secondary_y=False)
fig.update_yaxes(title_text="val perplexity", secondary_y=True)
fig.update_layout(title="Scheduled QAT — three schedules, INT4 target (micro + macro)",
                  height=480, hovermode="x unified",
                  margin=dict(t=80, b=40, l=60, r=60))
fig.show()

## 8. (Optional) Best schedule → INT8

If you have time left in the session, re-run the winning schedule at INT8 for the cross-method comparison in notebook 07. Set `RUN_BEST_INT8=True` in the override cell above.

In [ ]:
results_int8 = None
if RUN_BEST_INT8:
    src = f"configs/scheduled_qat/scheduled_{best}_int8.yaml"
    dst = KAGGLE_CFG / f"scheduled_{best}_int8.yaml"
    patch_sqat_config(src, dst)
    print(f"Running best schedule={best} at INT8...\n")
    results_int8 = run_experiment(
        config_path=str(dst),
        device_str="cuda",
        baseline_logits=str(FP32_LOGITS),
        run_benchmarks=False,
    )
    print(f"\nScheduled QAT ({best}) INT8:")
    for k, v in results_int8.items():
        print(f"  {k:25s}  {v}")
    gc.collect(); torch.cuda.empty_cache()
else:
    print("Skipped (RUN_BEST_INT8=False).")

## 9. Persist all results

In [ ]:
summary_path = Path(REPO_DIR) / "results" / "scheduled_qat_summary.json"
summary_path.parent.mkdir(parents=True, exist_ok=True)
summary = {
    "int4": {
        "linear": results_linear,
        "cosine": results_cosine,
        "step":   results_step,
    },
    "best_schedule": best,
    "int8_best": results_int8,
    "fp32": fp32,
}
with summary_path.open("w") as f:
    json.dump(summary, f, indent=2, default=str)
print(f"Wrote {summary_path}")
!ls -lh {REPO_DIR}/models/checkpoints/

## 10. Sample inference — FP32 vs each schedule

Same prompts and generator helper as notebooks 02/03 — direct cross-notebook comparison. With INT4 quantization, generation quality between schedules can differ noticeably even when their PPL/KLD numbers look similar; the schedule that handles the precision drop better tends to produce more coherent completions on the longer narrative prompts.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from src.models.model_wrapper import build_model_for_training
from src.utils.config_loader import load_config
from src.quantization.scheduled_qat import load_checkpoint as load_sqat_ckpt

SAMPLE_PROMPTS = [
    "The capital of France is",
    "Python is a programming language used for",
    "Once upon a time, in a small village,",
    "The chemical symbol for gold is",
    "To improve your sleep quality, you should",
]
MAX_NEW_TOKENS = 30
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-1.7B", cache_dir=MODEL_CACHE)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def generate_with_model(model, prompts=SAMPLE_PROMPTS, max_new_tokens=MAX_NEW_TOKENS):
    model.eval()
    out = []
    for p in prompts:
        ids = tokenizer(p, return_tensors="pt").input_ids.to(device)
        with torch.no_grad():
            gen = model.generate(
                ids, max_new_tokens=max_new_tokens, do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
        out.append(tokenizer.decode(gen[0][ids.shape[1]:], skip_special_tokens=True).strip())
    return out

def free():
    gc.collect(); torch.cuda.empty_cache()

# 1. FP32 baseline
print("Generating FP32 ...")
fp32_model = AutoModelForCausalLM.from_pretrained(
    "HuggingFaceTB/SmolLM2-1.7B", cache_dir=MODEL_CACHE, torch_dtype=torch.float16,
).to(device)
fp32_outs = generate_with_model(fp32_model)
del fp32_model; free()

# 2-4. Three Scheduled QAT INT4 variants
schedule_outs = {}
for name in ("linear", "cosine", "step"):
    print(f"Generating Scheduled QAT {name} INT4 ...")
    cfg = load_config(str(kaggle_cfgs[name])); cfg.model.cache_dir = MODEL_CACHE
    wrap = build_model_for_training(cfg, device,
                                     total_steps=int(cfg.training.epochs) * 1000)
    ckpt = f"{WORKING_DIR}/models/checkpoints/scheduled_qat_{name}_int4/final.pt"
    if Path(ckpt).exists():
        load_sqat_ckpt(wrap.model, ckpt)
    else:
        print(f"  WARNING — checkpoint missing: {ckpt}")
    schedule_outs[name] = generate_with_model(wrap.model)
    del wrap; free()

print("Done.")

In [ ]:
import pandas as pd
from IPython.display import display, HTML

inference_df = pd.DataFrame({
    "Prompt":      SAMPLE_PROMPTS,
    "FP32":        fp32_outs,
    "linear INT4": schedule_outs["linear"],
    "cosine INT4": schedule_outs["cosine"],
    "step INT4":   schedule_outs["step"],
})

inference_df.to_json(Path(REPO_DIR) / "results" / "scheduled_qat_inference.json",
                     orient="records", indent=2)

display(HTML(inference_df.to_html(index=False, escape=False).replace(
    "<table", '<table style="font-family:monospace; font-size:11px"')))

## Export everything to Drive

Same pattern as notebook 03 — checkpoints renamed (`scheduled_qat_<schedule>_int4.pt`), data samples, metric summary CSV, per-step + macro logs. Saved under `/content/drive/MyDrive/sqat-scheduled-qat/`.

In [ ]:
import shutil
import pandas as pd

dst = Path(DRIVE_ROOT)
(dst / "models" / "checkpoints").mkdir(parents=True, exist_ok=True)
(dst / "data_samples").mkdir(parents=True, exist_ok=True)

# 1. Rename + copy each schedule's checkpoint
for name in ("linear", "cosine", "step"):
    src = Path(REPO_DIR) / "models" / "checkpoints" / f"scheduled_qat_{name}_int4" / "final.pt"
    out = dst / "models" / "checkpoints" / f"scheduled_qat_{name}_int4.pt"
    if src.exists():
        shutil.copy2(src, out)
        print(f"  {out}  ({out.stat().st_size / 1e6:.0f} MB)")
    else:
        print(f"  SKIP — checkpoint not found: {src}")

if RUN_BEST_INT8 and results_int8:
    src = Path(REPO_DIR) / "models" / "checkpoints" / f"scheduled_qat_{best}_int8" / "final.pt"
    out = dst / "models" / "checkpoints" / f"scheduled_qat_{best}_int8.pt"
    if src.exists():
        shutil.copy2(src, out)
        print(f"  {out}  ({out.stat().st_size / 1e6:.0f} MB)")

# 2. Dataset samples (using cosine config — same data for all three schedules)
from src.utils.config_loader import load_config
from src.utils.data_loader import build_dataloaders, build_validation_loader

cfg = load_config(str(kaggle_cfgs["cosine"]))
train_loader, eval_loader = build_dataloaders(cfg, num_workers=0)
val_loader = build_validation_loader(cfg, num_workers=0)

def sample_loader(loader, n=8):
    chunks = []
    for batch in loader:
        chunks.append(batch["input_ids"][:n].cpu().clone())
        if sum(c.size(0) for c in chunks) >= n:
            break
    return torch.cat(chunks, dim=0)[:n]

torch.save({"input_ids": sample_loader(train_loader), "split": "train", "seq_length": cfg.data.seq_length},
           dst / "data_samples" / "train_sample.pt")
torch.save({"input_ids": sample_loader(val_loader),   "split": "validation", "seq_length": cfg.data.seq_length},
           dst / "data_samples" / "val_sample.pt")
torch.save({"input_ids": sample_loader(eval_loader),  "split": "test", "seq_length": cfg.data.seq_length},
           dst / "data_samples" / "test_sample.pt")
print(f"  Dataset samples written")

# 3. Metric summary CSV
metric_rows = []
for name, r in [("linear", results_linear), ("cosine", results_cosine), ("step", results_step)]:
    metric_rows.append({
        "experiment":      f"scheduled_qat_{name}_int4",
        "schedule":        name,
        "perplexity":      r.get("perplexity"),
        "kl_divergence":   r.get("kl_divergence"),
        "final_loss":      r.get("final_loss"),
        "training_time_s": r.get("training_time_seconds"),
        "total_steps":     r.get("total_steps"),
    })
metric_df = pd.DataFrame(metric_rows).round(6)
metric_csv = dst / "results" / "metric_summary.csv"
metric_csv.parent.mkdir(parents=True, exist_ok=True)
metric_df.to_csv(metric_csv, index=False)
print(f"  Metric summary -> {metric_csv}")
display(metric_df)

# 4-5. Copy results dir + configs
shutil.copytree(f"{REPO_DIR}/results", dst / "results", dirs_exist_ok=True)
shutil.copytree(COLAB_CFG, dst / "configs_colab" / "scheduled_qat", dirs_exist_ok=True)

print(f"\nAll outputs saved to: {dst}")
!du -sh {dst}